In [ ]:
print("="*70)
print("ArbItro - Ensemble Test  (Pipeline 1 + Pipeline 2)")
print("="*70)

import tensorflow as tf
print(f"\nTensorFlow version: {tf.__version__}")

gpu_name = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
if gpu_name:
    gpu_info = gpu_name[0].split(',')
    print(f"  GPU  : {gpu_info[0].strip()}")
    print(f"  VRAM : {gpu_info[1].strip()}")
    print("\n  System Resources:")
    !free -h | grep Mem
    !nvidia-smi --query-gpu=memory.free,memory.total --format=csv
else:
    print("\n  No GPU found")

print("\n" + "="*70)

In [ ]:
import os, sys

try:
    from google.colab import drive
    USE_COLAB = True
    drive.mount('/content/drive')
    BASE_DIR        = '/content/drive/MyDrive/ArbItro'
    P2_CODE_DIR     = os.path.join(BASE_DIR, 'ArbItro_code')
    P1_CODE_DIR     = os.path.join(BASE_DIR, 'ArbItro_code', 'pipeline1')
    DATASET_DIR     = os.path.join(BASE_DIR, 'ArbItro_dataset', 'dataset')
    TRAINING_ROOT   = os.path.join(BASE_DIR, 'ArbItro_Training')
except ImportError:
    USE_COLAB = False
    BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), '../..'))
    P2_CODE_DIR   = os.path.join(BASE_DIR, 'model', 'src', 'pipeline2')
    P1_CODE_DIR   = os.path.join(BASE_DIR, 'model', 'src', 'pipeline1')
    DATASET_DIR   = os.path.join(BASE_DIR, 'dataset')
    TRAINING_ROOT = os.path.join(BASE_DIR, 'ArbItro_Training')

sys.path.insert(0, P2_CODE_DIR)
sys.path.insert(1, P1_CODE_DIR)

print(f"  USE_COLAB    : {USE_COLAB}")
print(f"  P2_CODE_DIR  : {P2_CODE_DIR}")
print(f"  P1_CODE_DIR  : {P1_CODE_DIR}")
print(f"  DATASET_DIR  : {DATASET_DIR}")
print(f"  TRAINING_ROOT: {TRAINING_ROOT}")

required_paths = [
    os.path.join(DATASET_DIR, 'test', 'annotations.json'),
    os.path.join(DATASET_DIR, 'test'),
]
print("\nVerifying test paths:")
for path in required_paths:
    status = "Found" if os.path.exists(path) else "Not found"
    print(f"  {status}: {path}")

In [ ]:
!pip install opencv-python-headless

In [ ]:
import shutil

WORK_DIR = '/content/arbitro_model' if USE_COLAB else os.path.join(BASE_DIR, '.arbitro_work')
os.makedirs(WORK_DIR, exist_ok=True)

shutil.copy(os.path.join(P2_CODE_DIR, 'data_loader.py'), os.path.join(WORK_DIR, 'data_loader_p2.py'))
shutil.copy(os.path.join(P2_CODE_DIR, 'model.py'),       os.path.join(WORK_DIR, 'model_p2.py'))
shutil.copy(os.path.join(P1_CODE_DIR, 'data_loader.py'), os.path.join(WORK_DIR, 'data_loader_p1.py'))
shutil.copy(os.path.join(P1_CODE_DIR, 'model.py'),       os.path.join(WORK_DIR, 'model_p1.py'))

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print(f"  Working dir: {WORK_DIR}")
for f in sorted(os.listdir(WORK_DIR)):
    print(f"  {f}")

In [ ]:
try:
    import data_loader_p2 as _dl_p2
    import model_p2       as _m_p2
    Pipe2Generator                = _dl_p2.ArbItroDataGenerator
    BinaryBalancedAccuracy_P2     = _m_p2.BinaryBalancedAccuracy
    MulticlassBalancedAccuracy_P2 = _m_p2.MulticlassBalancedAccuracy
    print("  Pipeline 2 imports successful!")
except Exception as e:
    print(f"  Import error (Pipeline 2): {e}")
    raise

try:
    import data_loader_p1 as _dl_p1
    import model_p1       as _m_p1
    Pipe1Generator                = _dl_p1.ArbItroDataGenerator
    BinaryBalancedAccuracy_P1     = _m_p1.BinaryBalancedAccuracy
    MulticlassBalancedAccuracy_P1 = _m_p1.MulticlassBalancedAccuracy
    print("  Pipeline 1 imports successful!")
except Exception as e:
    print(f"  Import error (Pipeline 1): {e}")
    raise

In [ ]:
TEST_CONFIG = {
    'json_path'        : os.path.join(DATASET_DIR, 'test', 'annotations.json'),
    'video_dir'        : os.path.join(DATASET_DIR, 'test'),
    'pipe2_model_path' : os.path.join(TRAINING_ROOT, 'models', 'pipeline2.keras'),
    'pipe1_model_path' : os.path.join(TRAINING_ROOT, 'models', 'pipeline1.keras'),
    'dim'              : (224, 398),
    'n_frames'         : 16,
    'max_clips'        : 4,
    'batch_size'       : 28,
    'shuffle'          : False,
}

_OFF_MAP = {"No offence": 0, "": 0, "Between": 1, "Offence": 1}
_ACT_MAP = {
    "": 0, "Dont know": 0, "Dive": 0,
    "Challenge": 1, "Tackling": 1, "Standing tackling": 1,
    "Holding": 2, "Pushing": 2, "Elbowing": 2,
    "High leg": 3,
}
MAP_OFF = {0: "No Offence",   1: "Offence"}
MAP_SEV = {0: "No Card",      1: "Yellow",      2: "Red"}
MAP_ACT = {0: "Neutral/Dive", 1: "Tackles",     2: "Upper Body", 3: "High Leg"}

def _get_sev(row):
    try:
        v = float(row.get('Severity', 0))
        if v >= 5.0: return 2
        if v >= 3.0: return 1
        return 0
    except: return 0

for key in ('pipe2_model_path', 'pipe1_model_path'):
    status = "Found" if os.path.exists(TEST_CONFIG[key]) else "Not found"
    print(f"  {status}: {TEST_CONFIG[key]}")